In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn.linear_model as lm
import pandas as pd

import pickle

import mat73

import sys
sys.path.append('/home/austin/Aggression/Code/NMF')
from nmf_elastic import NMF_logistic

sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data


In [ ]:
fnm='CL_baseline_all_validate3.mat'
features = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])
power = features[0]
coherence = features[1]
granger = features[2]

In [ ]:
print(type(features[0]))
print(type(features[1]))
print(type(features[2]))


## Project into the global model

In [ ]:
# Process data identically
granger = np.exp(granger)
granger[granger>10] = 10
power = power*10
power[power>6] = 6

X = np.hstack((power,coherence,granger))

In [ ]:
nFact = 8
myDict = {}

nIter = 20000
model = NMF_logistic(nFact,nIter=nIter,LR=1e-3,mu=1.0,batchSize=100)

my_dict = pickle.load(open('../Unbalanced_Elastic_12_enc_1.0.p','rb'))
model.A_enc = my_dict['A_enc']
model.B_enc = my_dict['B_enc']

In [ ]:
S_test = model.transform(X)

In [ ]:
np.savetxt('CL_baseline_global.csv',S_test,delimiter=',',fmt='%0.8f')

## Project into  the individual models

In [ ]:
import cloudpickle
model_list = cloudpickle.load(open('../SingleRegionModels/AggresionPro.p','rb'))

In [ ]:
model_list.keys()

In [ ]:
for i in range(11):
    S_test_single = model_list['models'][i].transform(power[:,i*56:(i+1)*56])
    sname = 'CL_baseliene_%d.csv'%int(i)
    np.savetxt(sname,S_test_single,delimiter=',',fmt='%0.8f')